<a href="https://colab.research.google.com/github/Adyypower/Deep-learning-Models-or-topics/blob/main/GPT_for_Instruction_Following.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 **Training GPT for Instruction Following**

In this notebook, we demonstrate the process of training a GPT model to follow instructions effectively, using a custom dataset for varied tasks. Let's embark on this journey from data preprocessing to model training and text generation.


## Setup and Dependencies

First, we import necessary libraries and set up our environment to handle the tasks.

In [ ]:
!pip install transformers[torch]==4.38.2
!pip install datasets===2.13.1

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset



## Data Exploration and Preparation 📊

Let's load and preview our dataset, ensuring we understand the kind of data we're working with.


## Data Exploration and Preparation 📊

Let's load and preview our dataset, ensuring we understand the kind of data we're working with.

In [ ]:

dataset = load_dataset("hakurei/open-instruct-v1", split='train')
dataset.to_pandas().sample(20)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

instruct_data.json:   0%|          | 0.00/104M [00:00<?, ?B/s]

additional_data.json:   0%|          | 0.00/19.7M [00:00<?, ?B/s]

alpaca_data.json:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

...

self_instruct.json:   0%|          | 0.00/26.2M [00:00<?, ?B/s]

sharegpt_data.json:   0%|          | 0.00/109M [00:00<?, ?B/s]

synthetic_instruct.json:   0%|          | 0.00/19.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/498813 [00:00<?, ? examples/s]

,output,input,instruction
23955,Two ingredients that could be added to a smoot...,,Please suggest two possible ingredients that c...
89257,Neutral,,Detect the emotion from the given text.\n\nI a...
412687,"""Damn it feels good to be a hacker""\n\nHacking...",,"Write a parody of ""Damn it feels good to be a ..."
390061,Action,,"Classify whether a movie is action, comedy, dr..."
33826,Former Indian Prime Minister Rajiv Gandhi and ...,,Name a famous person with roots in the city of...
224355,for row in A:\n for elem in row:\n p...,,Write a code to print out a two-dimensional ar...
243469,There are 500 episodes of Naruto: Shippuden.,Naruto was serialized in Shueisha's shōnen man...,"From the passage provided, extract the number ..."
134310,- Companies could be using social media to pro...,,Give examples of how companies could be using ...
333734,A shield with a lion on it. The lion is holdin...,,Design a coat of arms for the given name.\n\nJ...
42444,"Knock knock, the doorbell's broken so I'm usin...","Knock knock, who's there?",Generate a new joke by editing the following s...


In [ ]:

def preprocess(example):
    example['prompt'] = f"{example['instruction']} {example['input']} {example['output']}"

    return example

def tokenize_datasets(dataset):
    tokenized_dataset = dataset.map(lambda example: tokenizer(example['prompt'], truncation=True, max_length=128), batched=True, remove_columns=['prompt'])

    return tokenized_dataset




### Shuffling and Splitting the Dataset

Next, we shuffle the dataset and split it into training and test sets to ensure robust model training and evaluation.

In [ ]:
dataset = dataset.map(preprocess, remove_columns=['instruction', 'input', 'output'])
dataset =  dataset.shuffle(42).select(range(100000)).train_test_split(test_size=0.1, seed=42)

Map:   0%|          | 0/498813 [00:00<?, ? examples/s]

In [ ]:

train_dataset = dataset['train']
test_dataset = dataset['test']

## Model Initialization and Tokenization 🚀

We set up the tokenizer and the model, ensuring that our tokens align with the model's expected format.

In [ ]:

MODEL_NAME = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

train_dataset = tokenize_datasets(train_dataset)
test_dataset = tokenize_datasets(test_dataset)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Training the GPT Model 🎯
Now, we configure the training parameters and initiate the training process using our prepared datasets.

In [ ]:

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

traing_args = TrainingArguments(output_dir="models/diablo_gpt",
                                num_train_epochs=1,
                                per_device_train_batch_size=32,
                                per_device_eval_batch_size=32)\

trainer = Trainer(model=model,
                    args=traing_args,
                    train_dataset=train_dataset,
                    eval_dataset=test_dataset,
                    data_collator=data_collator)


In [ ]:
trainer.train() # this will take a long time


NameError: name 'trainer' is not defined

In [ ]:
# Get the trained checkpoint directly
model = AutoModelForCausalLM.from_pretrained("TheFuzzyScientist/diabloGPT_open-instruct")

config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

## Text Generation and Application 📝
Finally, let's demonstrate the capability of our trained model by generating responses to various instructions.

In [ ]:

def generate_text(prompt):
    inputs = tokenizer.encode(prompt, return_tensors='pt') #.to("cuda") # <-- if running on GPU, uncomment this
    outputs = model.generate(inputs, max_length=64, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return generated[:generated.rfind('.')+1]

In [ ]:
generate_text("What's the best way to cook chiken breast?")

"What's the best way to cook chiken breast?  The best way to cook chiken breast is to season it with salt and pepper, then heat a pan over medium heat. Add a tablespoon of olive oil and cook for about 5 minutes, stirring occasionally."

In [ ]:
generate_text("Should I invest stocks?")


'Should I invest stocks?  Yes, it is a good idea to invest in stocks. It is important to understand the risks associated with investing in stocks and to make sure that you are taking the necessary precautions. It is also important to understand the potential returns and to make sure that you are making the right investment.'

In [ ]:
generate_text("I need a place to go for this summer vacation, what locations would you recommend")


'I need a place to go for this summer vacation, what locations would you recommend.  I would recommend visiting the beach in San Diego, California. It is a popular destination for vacationers and has a great view of the ocean.'

In [ ]:
generate_text("What's the fastest route from NY City to Boston?")


"What's the fastest route from NY City to Boston?  The fastest route from New York City to Boston is by taking the New York City subway. The subway takes about 3 hours and 15 minutes to get from the city center to the Boston Common."


## Conclusion and Further Exploration 🌟

This notebook guides you through training a GPT model to follow complex instructions. Experiment further by altering the dataset or tweaking the model and training configurations to explore new possibilities!